# Getting Started: H2 VQE Noise Scans

This notebook scans the five built-in single-qubit noise channels supported by the VQE package for **dihydrogen (`H2`)**:

- depolarizing
- amplitude damping
- phase damping
- bit flip
- phase flip

Each scan uses the same grid of probabilities:
`0.00, 0.01, 0.02, ..., 0.10`.

Unit contract used throughout the package:

- coordinate inputs may be given in `angstrom` or `bohr`
- the chosen `unit` affects geometry only
- energies in this notebook are always reported in Hartree (`Ha`)

The goal is to compare how each noise type shifts the final VQE energy and how the convergence trace changes as noise increases.


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from common.hamiltonian import get_exact_spectrum
from vqe import run_vqe


## Reference Energy

We use the exact ground-state energy of `H2` as the baseline for all error calculations.


In [ ]:
exact_spectrum = np.asarray(get_exact_spectrum("H2"), dtype=float)
exact_ground_energy = float(np.min(exact_spectrum))
exact_ground_energy


## Fixed VQE Settings

These settings are reused across every scan so that the only thing changing is the selected noise probability.


In [ ]:
molecule = "H2"
ansatz_name = "UCCSD"
optimizer_name = "Adam"
steps = 60
stepsize = 0.2
seed = 0
mapping = "jordan_wigner"

noise_levels = np.round(np.linspace(0.00, 0.10, 11), 2)
noise_levels


## Noise Types

Each scan enables exactly one built-in noise channel and keeps the others fixed at zero.


In [ ]:
noise_specs = {
    "depolarizing": {
        "kwargs": lambda p: {
            "depolarizing_prob": float(p),
            "amplitude_damping_prob": 0.0,
            "phase_damping_prob": 0.0,
            "bit_flip_prob": 0.0,
            "phase_flip_prob": 0.0,
        },
        "title": "Depolarizing",
    },
    "amplitude_damping": {
        "kwargs": lambda p: {
            "depolarizing_prob": 0.0,
            "amplitude_damping_prob": float(p),
            "phase_damping_prob": 0.0,
            "bit_flip_prob": 0.0,
            "phase_flip_prob": 0.0,
        },
        "title": "Amplitude Damping",
    },
    "phase_damping": {
        "kwargs": lambda p: {
            "depolarizing_prob": 0.0,
            "amplitude_damping_prob": 0.0,
            "phase_damping_prob": float(p),
            "bit_flip_prob": 0.0,
            "phase_flip_prob": 0.0,
        },
        "title": "Phase Damping",
    },
    "bit_flip": {
        "kwargs": lambda p: {
            "depolarizing_prob": 0.0,
            "amplitude_damping_prob": 0.0,
            "phase_damping_prob": 0.0,
            "bit_flip_prob": float(p),
            "phase_flip_prob": 0.0,
        },
        "title": "Bit Flip",
    },
    "phase_flip": {
        "kwargs": lambda p: {
            "depolarizing_prob": 0.0,
            "amplitude_damping_prob": 0.0,
            "phase_damping_prob": 0.0,
            "bit_flip_prob": 0.0,
            "phase_flip_prob": float(p),
        },
        "title": "Phase Flip",
    },
}


## Scan Helper

This helper runs `run_vqe(...)` for a chosen noise type over the full probability grid and records both the final energy and the full convergence history.


In [ ]:
def run_noise_scan(noise_name: str):
    spec = noise_specs[noise_name]
    rows = []
    traces = {}

    for p in noise_levels:
        result = run_vqe(
            molecule=molecule,
            ansatz_name=ansatz_name,
            optimizer_name=optimizer_name,
            steps=int(steps),
            stepsize=float(stepsize),
            seed=int(seed),
            mapping=mapping,
            noisy=bool(p > 0.0),
            plot=False,
            force=False,
            **spec["kwargs"](p),
        )

        energies = np.asarray(result["energies"], dtype=float)
        final_energy = float(result["energy"])
        traces[float(p)] = energies
        rows.append(
            {
                "noise_type": noise_name,
                "noise_label": spec["title"],
                "probability": float(p),
                "final_energy": final_energy,
                "abs_error": abs(final_energy - exact_ground_energy),
                "iterations": int(len(energies)),
            }
        )

    return pd.DataFrame(rows), traces


## Run All Scans


In [ ]:
scan_frames = []
scan_traces = {}

for noise_name in noise_specs:
    frame, traces = run_noise_scan(noise_name)
    scan_frames.append(frame)
    scan_traces[noise_name] = traces

scan_df = pd.concat(scan_frames, ignore_index=True)
scan_df


## Summary Table


In [ ]:
summary_df = scan_df.copy()
summary_df["final_energy"] = summary_df["final_energy"].map(lambda x: f"{x:+.8f}")
summary_df["abs_error"] = summary_df["abs_error"].map(lambda x: f"{x:.8f}")
summary_df.pivot(index="probability", columns="noise_label", values="final_energy")


## Final Energy vs Noise Strength

All scan curves use the same marker style. These are scan plots, so markers are kept; the convergence plots below use continuous lines only.


In [ ]:
plt.figure(figsize=(10, 5))
for noise_name, group in scan_df.groupby("noise_label", sort=False):
    plt.plot(
        group["probability"],
        group["final_energy"],
        marker="o",
        lw=2,
        label=noise_name,
    )

plt.axhline(exact_ground_energy, color="black", ls="--", lw=1.5, label="Exact")
plt.xlabel("Noise probability")
plt.ylabel("Final VQE energy (Ha)")
plt.title("H2 VQE noise scans: final energy")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## Absolute Error vs Noise Strength


In [ ]:
plt.figure(figsize=(10, 5))
for noise_name, group in scan_df.groupby("noise_label", sort=False):
    plt.plot(
        group["probability"],
        group["abs_error"],
        marker="o",
        lw=2,
        label=noise_name,
    )

plt.xlabel("Noise probability")
plt.ylabel("|E_VQE - E_exact| (Ha)")
plt.title("H2 VQE noise scans: absolute error")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## Convergence Traces at Representative Noise Levels

For each noise type, we compare the optimization trace at probabilities `0.00`, `0.05`, and `0.10`. These are convergence plots, so they use continuous lines without markers.


In [ ]:
selected_levels = [0.00, 0.05, 0.10]
fig, axes = plt.subplots(3, 2, figsize=(12, 12), sharex=True, sharey=True)
axes = axes.flatten()

for ax, (noise_name, spec) in zip(axes, noise_specs.items()):
    for p in selected_levels:
        energies = scan_traces[noise_name][float(np.round(p, 2))]
        ax.plot(range(len(energies)), energies, lw=2, label=f"p={p:.2f}")
    ax.axhline(exact_ground_energy, color="black", ls="--", lw=1.0)
    ax.set_title(spec["title"])
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Energy (Ha)")
    ax.grid(True, alpha=0.3)
    ax.legend()

if len(noise_specs) < len(axes):
    for ax in axes[len(noise_specs):]:
        ax.axis("off")

fig.suptitle("H2 VQE convergence under built-in noise channels", y=1.02)
fig.tight_layout()
plt.show()


## Quick Takeaways

- All scans use the exact same optimizer, ansatz, seed, and probability grid.
- Each curve isolates one built-in noise channel at a time.
- The convergence panel makes it easier to see whether a noise type mainly shifts the endpoint, slows optimization, or both.
